In [1]:
# Install required packages
!pip install -q sae-lens transformer-lens transformers scipy pandas matplotlib seaborn tqdm openai
!pip uninstall numpy -y
!pip install numpy==1.26.4 -q
!pip install --force-reinstall pandas -q
pip install torchvision --upgrade

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.4.1+cu124 requires torch==2.4.1, but you have torch 2.10.0 which is incompatible.
torchvision 0.19.1+cu124 requires torch==2.4.1, but you have torch 2.10.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.19.1+cu124 requires torch==2.4.1, but you have torch 2.10.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
ERROR: pip

In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from tqdm.auto import tqdm
import re
import gc
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForCausalLM
from sae_lens import SAE  # adjust if you use a different SAE library

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

Device: cuda
GPU: NVIDIA B200


In [2]:
from huggingface_hub import login
login()

In [3]:
from sae_lens import SAE
import transformer_lens

print("Loading Gemma 9B IT model with TransformerLens...")
print("This will take a few minutes...")

tl_model = transformer_lens.HookedTransformer.from_pretrained(
    "google/gemma-2-9b-it",
    device="cuda",
    dtype=torch.bfloat16,
)

print("Model loaded successfully!")
print(f"Model has {tl_model.cfg.n_layers} layers")
print(f"Model dimension: {tl_model.cfg.d_model}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Gemma 9B IT model with TransformerLens...
This will take a few minutes...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-9b-it into HookedTransformer
Model loaded successfully!
Model has 42 layers
Model dimension: 3584


In [4]:
print("Loading SAE from Gemma Scope (Layer 20)...")

sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-9b-it-res-canonical",
    sae_id="layer_20/width_131k/canonical",  # Using 131k width for layer 20 features
    device="cuda"
)

print(f"SAE loaded successfully!")
print(f"SAE width: {sae.cfg.d_sae}")
print(f"SAE input dimension: {sae.cfg.d_in}")

Loading SAE from Gemma Scope (Layer 20)...
SAE loaded successfully!
SAE width: 131072
SAE input dimension: 3584


In [5]:
# ============================================================================
# UNIVERSAL FEATURES (All 3 Languages - Layer 20)
# ============================================================================
UNIVERSAL_FEATURES = [
    3249, 2870, 7789, 667,
    3375, 14220, 16354, 8515, 12662, 3542,
    35440, 93521, 97003, 33236, 88782, 92777,
    20069, 71479, 24006, 46374,
]

# ============================================================================
# LANGUAGE-SPECIFIC TOP FEATURES
# ============================================================================
ENGLISH_FEATURES = [
    35440, 33236, 93521, 7697, 108864,
    21876, 28165, 40141, 88782, 103326,
]

HEBREW_FEATURES = [
    115163, 35440, 74327, 93521, 97003,
    30218, 7163, 8408, 104622, 86719,
]

RUSSIAN_FEATURES = [
    93521, 97003, 115163, 45410, 35440,
    83575, 92777, 88782, 3598, 24006,
]

# ============================================================================
# CROSS-LANGUAGE EXCLUSIVE PAIRS
# ============================================================================
HEBREW_RUSSIAN_EXCLUSIVE = [
    115163, 45410, 8408, 30218, 46695,
    104622, 51811, 39066, 105496, 46769,
    98848, 72807, 122872,
]

HEBREW_ENGLISH_EXCLUSIVE = [
    85416, 47814, 23902, 61806, 98997,
]

ENGLISH_RUSSIAN_EXCLUSIVE = [
    21876, 93245, 129197, 27619, 107295,
    128449, 73106, 67248, 70055, 68324, 111712,
]

FEATURE_SETS = {
    'universal':    UNIVERSAL_FEATURES,
    'english':      ENGLISH_FEATURES,
    'hebrew':       HEBREW_FEATURES,
    'russian':      RUSSIAN_FEATURES,
    'heb_rus_excl': HEBREW_RUSSIAN_EXCLUSIVE,
    'heb_eng_excl': HEBREW_ENGLISH_EXCLUSIVE,
    'eng_rus_excl': ENGLISH_RUSSIAN_EXCLUSIVE,
}

print("Feature Sets Defined:")
for name, feats in FEATURE_SETS.items():
    print(f"  {name:16s}: {len(feats):3d} features")
all_feats = set(f for fs in FEATURE_SETS.values() for f in fs)
print(f"\nTotal unique features: {len(all_feats)}")

Feature Sets Defined:
  universal       :  20 features
  english         :  10 features
  hebrew          :  10 features
  russian         :  10 features
  heb_rus_excl    :  13 features
  heb_eng_excl    :   5 features
  eng_rus_excl    :  11 features

Total unique features: 59


In [6]:
DATA_DIR = "/workspace"  

import os

# ── Helper: peek at columns ─────────────────────────────────────────────────
csv_files = [
    "unified_english_dataset.csv",
    "hebrew_slang_dataset.csv",
    "negatives_with_term.csv",
    "russian_slang_positive_results.csv",
    "russian_literal_negatives_dataset.csv",
    "hebrew_slang_classification_results.csv",
]

for f in csv_files:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        df_tmp = pd.read_csv(path, nrows=3)
        print(f"\n{f}  — columns: {list(df_tmp.columns)}")
        display(df_tmp.head(2))
    else:
        print(f"\n[MISSING] {f}")


unified_english_dataset.csv  — columns: ['term', 'sentence', 'source', 'classification', 'model_response']


,term,sentence,source,classification,model_response
0,jam,"If you get into a jam, you can call me.",opensub,Slang,Slang
1,crack,I want to have a crack at your boss.,opensub,Slang,Slang



hebrew_slang_dataset.csv  — columns: ['sentence', 'is_slang', 'slang_words', 'dataset']


,sentence,is_slang,slang_words,dataset
0,אתה לא נשיא במקרה ? ? ? ? כן זה שהתפקיד שלו זה...,1,אש,sepidmnorozy/Hebrew_sentiment
1,אש אתה ! ! !,1,אש,sepidmnorozy/Hebrew_sentiment



negatives_with_term.csv  — columns: ['SENTENCE', 'FULL_CONTEXT', 'MOVIE_ID', 'SENT_ID', 'REGION', 'YEAR', 'term']


...

,slang_term,text,literal_meaning,slang_meaning,confidence,gpt_explanation,likes,date,link,owner_id,post_id,gpt_sense,classification,model_response
0,огонь,"Дубленка, эко-кожа\nРазмеры 42-44-46-48\nНовый...",fire (flames),awesome/amazing/great,HIGH,«Бомба» используется для выражения высокой оце...,0,2025-10-19,https://vk.com/wall-218146802_137294,-218146802,137294,slang,Yes,Yes
1,огонь,"Такие, как Red Lady, зажгут твой огонь внутри!...",fire (flames),awesome/amazing/great,HIGH,«Огонь» здесь используется в значении «отличны...,0,2025-10-19,https://vk.com/wall43632392_5957,43632392,5957,slang,Yes,Yes



russian_literal_negatives_dataset.csv  — columns: ['slang_term', 'text', 'literal_meaning', 'slang_meaning', 'usage_type', 'confidence', 'gpt_explanation', 'likes', 'date', 'link', 'owner_id', 'post_id']


,slang_term,text,literal_meaning,slang_meaning,usage_type,confidence,gpt_explanation,likes,date,link,owner_id,post_id
0,огонь,⚡️ Пожар тушили на ферме в Холмском районе\n\n...,fire (flames),awesome/amazing/great,LITERAL,HIGH,"In this sentence, ""огонь"" refers to actual fla...",0,2025-12-08,https://vk.com/wall-233899072_336,-233899072,336
1,огонь,"Ты, кажется, искал здесь? Не ищи.\nГремит засо...",fire (flames),awesome/amazing/great,LITERAL,HIGH,"In the sentence, ""огонь"" refers to the literal...",0,2025-12-08,https://vk.com/wall443046507_332,443046507,332



hebrew_slang_classification_results.csv  — columns: ['sentence', 'is_slang', 'slang_words', 'dataset', 'classification']


,sentence,is_slang,slang_words,dataset,classification
0,אתה לא נשיא במקרה ? ? ? ? כן זה שהתפקיד שלו זה...,1,אש,sepidmnorozy/Hebrew_sentiment,No
1,אש אתה ! ! !,1,אש,sepidmnorozy/Hebrew_sentiment,Yes


In [7]:
def load_and_unify(data_dir):
    dfs = []

    # ── ENGLISH ─────────────────────────────────────────────────────────────
    path = os.path.join(data_dir, "unified_english_dataset.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df.rename(columns={'classification': 'label'})[['sentence', 'term', 'label']]
        df['language'] = 'en'
        dfs.append(df)
        print(f"English: {len(df)} rows ({df['label'].value_counts().to_dict()})")

    # ── HEBREW SLANG ────────────────────────────────────────────────────────
    path = os.path.join(data_dir, "hebrew_slang_dataset.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df.rename(columns={'slang_words': 'term'})[['sentence', 'term']]
        df['label'] = 'Slang'
        df['language'] = 'he'
        dfs.append(df)
        print(f"Hebrew slang: {len(df)} rows")

    # ── HEBREW LITERAL ──────────────────────────────────────────────────────
    path = os.path.join(data_dir, "negatives_with_term.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df.rename(columns={'SENTENCE': 'sentence'})[['sentence', 'term']]
        df['label'] = 'Literal'
        df['language'] = 'he'
        dfs.append(df)
        print(f"Hebrew literal: {len(df)} rows")

    # ── RUSSIAN SLANG ───────────────────────────────────────────────────────
    path = os.path.join(data_dir, "russian_slang_positive_results.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df.rename(columns={'text': 'sentence', 'slang_term': 'term'})[['sentence', 'term']]
        df['label'] = 'Slang'
        df['language'] = 'ru'
        dfs.append(df)
        print(f"Russian slang: {len(df)} rows")

    # ── RUSSIAN LITERAL ─────────────────────────────────────────────────────
    path = os.path.join(data_dir, "russian_literal_negatives_dataset.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df.rename(columns={'text': 'sentence', 'slang_term': 'term'})[['sentence', 'term']]
        df['label'] = 'Literal'
        df['language'] = 'ru'
        dfs.append(df)
        print(f"Russian literal: {len(df)} rows")

    unified = pd.concat(dfs, ignore_index=True)
    return unified

data = load_and_unify(DATA_DIR)
print(f"\nUnified dataset: {len(data)} rows")
print(data.groupby(['language', 'label']).size().unstack(fill_value=0))

English: 2835 rows ({'Literal': 1857, 'Slang': 968, 'Unknown': 10})
Hebrew slang: 4366 rows
Hebrew literal: 2193 rows
Russian slang: 538 rows
Russian literal: 721 rows

Unified dataset: 10653 rows
label     Literal  Slang  Unknown
language                         
en           1857    968       10
he           2193   4366        0
ru            721    538        0


In [8]:
print("="*80)
print("EXTRACTING STEERING VECTORS FROM DISCOVERED FEATURES")
print("="*80)

decoder_weights = sae.W_dec.data  # shape: (n_features, d_model)
print(f"Decoder matrix shape: {decoder_weights.shape}")

steering_vectors = {}

for set_name, features in FEATURE_SETS.items():
    feature_vectors = []
    for feat_idx in features:
        if feat_idx < decoder_weights.shape[0]:
            feature_vectors.append(decoder_weights[feat_idx].clone())
        else:
            print(f"  Warning: Feature {feat_idx} out of bounds, skipping")
    if feature_vectors:
        mean_vector = torch.stack(feature_vectors).mean(dim=0)
        mean_vector = mean_vector / mean_vector.norm()
        steering_vectors[set_name] = mean_vector
        print(f"  {set_name.upper():16s} vector created from {len(feature_vectors)} features")

print(f"\n✓ Created {len(steering_vectors)} steering vectors")

EXTRACTING STEERING VECTORS FROM DISCOVERED FEATURES
Decoder matrix shape: torch.Size([131072, 3584])
  UNIVERSAL        vector created from 20 features
  ENGLISH          vector created from 10 features
  HEBREW           vector created from 10 features
  RUSSIAN          vector created from 10 features
  HEB_RUS_EXCL     vector created from 13 features
  HEB_ENG_EXCL     vector created from 5 features
  ENG_RUS_EXCL     vector created from 11 features

✓ Created 7 steering vectors


In [9]:
def create_prompt(sentence, term):
    """Few-shot prompt for Slang vs Literal classification."""
    prompt = f"""Classify whether the highlighted term in each sentence is used as slang or with its literal meaning.
Important: Respond with ONLY the classification inside the tags: [CLASSIFICATION_START]Slang[CLASSIFICATION_END] or [CLASSIFICATION_START]Literal[CLASSIFICATION_END]

Examples:

Sentence: "This song is straight fire"
Term: "fire"
Answer: [CLASSIFICATION_START]Slang[CLASSIFICATION_END]

Sentence: "The fire burned for 24 hours"
Term: "fire"
Answer: [CLASSIFICATION_START]Literal[CLASSIFICATION_END]

Sentence: "That party was lit"
Term: "lit"
Answer: [CLASSIFICATION_START]Slang[CLASSIFICATION_END]

Sentence: "She lit the candle carefully"
Term: "lit"
Answer: [CLASSIFICATION_START]Literal[CLASSIFICATION_END]

Sentence: "Your performance was sick"
Term: "sick"
Answer: [CLASSIFICATION_START]Slang[CLASSIFICATION_END]

Sentence: "He was sick with the flu"
Term: "sick"
Answer: [CLASSIFICATION_START]Literal[CLASSIFICATION_END]

Now classify:

Sentence: "{sentence}"
Term: "{term}"
Answer:"""
    return prompt


def parse_classification(text):
    """Extract classification from model output."""
    match = re.search(r'\[CLASSIFICATION_START\](.*?)\[CLASSIFICATION_END\]', text)
    if match:
        label = match.group(1).strip().capitalize()
        if label in ('Slang', 'Literal'):
            return label
    # Fallback: look for the words
    text_lower = text.lower()
    if 'slang' in text_lower and 'literal' not in text_lower:
        return 'Slang'
    elif 'literal' in text_lower and 'slang' not in text_lower:
        return 'Literal'
    return 'Unknown'


# Test
test_prompt = create_prompt("That movie was fire", "fire")
print("Prompt length:", len(test_prompt), "chars")
print("Parse test:", parse_classification("[CLASSIFICATION_START]Slang[CLASSIFICATION_END]"))

Prompt length: 997 chars
Parse test: Slang


In [10]:
def get_hook_layer_name(layer_idx):
    return f"blocks.{layer_idx}.hook_resid_post"

def classify_batch(
    sentences, terms,
    steering_vector=None, alpha=0.0,
    hook_layer=20,
    max_new_tokens=30,
    batch_size=300,
):
    results = []
    hook_point = get_hook_layer_name(hook_layer)

    def steering_hook(activation, hook):
        return activation + alpha * steering_vector.to(activation.device, activation.dtype)

    for i in range(0, len(sentences), batch_size):
        batch_sents = sentences[i:i+batch_size]
        batch_terms = terms[i:i+batch_size]

        prompts = [create_prompt(s, t) for s, t in zip(batch_sents, batch_terms)]
        encodings = tl_model.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        if steering_vector is not None and alpha != 0:
            tl_model.add_hook(hook_point, steering_hook)

        output_ids = None
        try:
            with torch.no_grad():
                output_ids = tl_model.generate(
                    encodings['input_ids'],
                    max_new_tokens=max_new_tokens,
                    temperature=0.0,
                )

            for j in range(len(batch_sents)):
                prompt_len = (encodings['input_ids'][j] != tl_model.tokenizer.pad_token_id).sum()
                generated = tl_model.tokenizer.decode(
                    output_ids[j][prompt_len:],
                    skip_special_tokens=True,
                )
                results.append(parse_classification(generated))
        finally:
            tl_model.reset_hooks()
            if output_ids is not None:
                del output_ids
            del encodings
            torch.cuda.empty_cache()

        print(f"    batch {i//batch_size + 1}/{(len(sentences) + batch_size - 1)//batch_size} done ({len(results)}/{len(sentences)})")

    return results

print("✓ Inference helpers defined")

✓ Inference helpers defined


In [11]:
# Optional: subsample for speed. Set to None to use full dataset.
MAX_PER_GROUP = None  # set to None for full dataset

if MAX_PER_GROUP is not None:
    data_sample = data.groupby(['language', 'label']).apply(
        lambda x: x.sample(min(len(x), MAX_PER_GROUP), random_state=42)
    ).reset_index(drop=True)
    print(f"Subsampled: {len(data_sample)} rows")
else:
    data_sample = data.copy()
    print(f"Using full dataset: {len(data_sample)} rows")

print(data_sample.groupby(['language', 'label']).size().unstack(fill_value=0))

Using full dataset: 10653 rows
label     Literal  Slang  Unknown
language                         
en           1857    968       10
he           2193   4366        0
ru            721    538        0


In [12]:
tl_model.tokenizer.pad_token = tl_model.tokenizer.eos_token
ALPHAS_POS = [50, 100, 150, 200]    # Literal → Slang
ALPHAS_NEG = [-50, -100, -150, -200]  # Slang → Literal

# Which steering vectors to test on which language datasets
# Key: (vector_name, language) → scientific question being asked
EXPERIMENT_MATRIX = {
    # === UNIVERSAL FEATURES ON ALL LANGUAGES ===
    ('universal', 'en'): 'Universal → English',
    ('universal', 'he'): 'Universal → Hebrew',
    ('universal', 'ru'): 'Universal → Russian',


    # === CROSS-LANGUAGE SINGLE ===
    ('english',  'he'): 'English → Hebrew',
    ('english',  'ru'): 'English → Russian',
    ('hebrew',   'en'): 'Hebrew → English',
    ('hebrew',   'ru'): 'Hebrew → Russian',
    ('russian',  'en'): 'Russian → English',
    ('russian',  'he'): 'Russian → Hebrew',

    # === PAIR-EXCLUSIVE → EXCLUDED LANGUAGE ===
    ('heb_rus_excl', 'en'): 'Heb+Rus (excl) → English',
    ('heb_eng_excl', 'ru'): 'Heb+Eng (excl) → Russian',
    ('eng_rus_excl', 'he'): 'Eng+Rus (excl) → Hebrew',

}

print(f"Total experiments: {len(EXPERIMENT_MATRIX)} vector×language pairs")
print(f"× {len(ALPHAS_POS)} positive alphas + {len(ALPHAS_NEG)} negative alphas")
print(f"= {len(EXPERIMENT_MATRIX) * (len(ALPHAS_POS) + len(ALPHAS_NEG))} steering runs")

Total experiments: 12 vector×language pairs
× 4 positive alphas + 4 negative alphas
= 96 steering runs


In [14]:
import gc

for bs in [10, 20, 30, 50, 80, 100, 150]:
    gc.collect()
    torch.cuda.empty_cache()
    
    test_sents = data_sample['sentence'].tolist()[:bs]
    test_terms = data_sample['term'].tolist()[:bs]
    
    prompts = [create_prompt(s, t) for s, t in zip(test_sents, test_terms)]
    encodings = tl_model.tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=512,
    ).to(device)
    
    try:
        with torch.no_grad():
            output_ids = tl_model.generate(
                encodings['input_ids'],
                max_new_tokens=30,
                temperature=0.0,
            )
        vram = torch.cuda.memory_allocated(0) / 1024**3
        print(f"  batch={bs:4d} ✅  VRAM: {vram:.1f} GB")
        del output_ids, encodings
        torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError:
        print(f"  batch={bs:4d} ❌  OOM")
        del encodings
        gc.collect()
        torch.cuda.empty_cache()
        break

  0%|          | 0/30 [00:00<?, ?it/s]

  batch=  10 ✅  VRAM: 150.9 GB


  0%|          | 0/30 [00:00<?, ?it/s]

  batch=  20 ✅  VRAM: 150.9 GB


  0%|          | 0/30 [00:00<?, ?it/s]

  batch=  30 ✅  VRAM: 150.9 GB


  0%|          | 0/30 [00:00<?, ?it/s]

  batch=  50 ❌  OOM


In [13]:
# ============================================================================
# RUN ALL STEERING EXPERIMENTS
# ============================================================================
print("CELL STARTED")
print(f"EXPERIMENT_MATRIX has {len(EXPERIMENT_MATRIX)} entries")
print(f"data_sample has {len(data_sample)} rows")
print(f"steering_vectors has {len(steering_vectors)} keys")
STEERING_CHECKPOINT = "/workspace/steering_results_checkpoint.csv"
BATCH_SIZE_STEERING = 300

# Resume from checkpoint if exists
if os.path.exists(STEERING_CHECKPOINT):
    results_records = pd.read_csv(STEERING_CHECKPOINT).to_dict('records')
    done_keys = set(
        (r['vector'], r['language'], r['direction'], r['alpha'])
        for r in results_records
    )
    print(f"✓ Loaded checkpoint: {len(results_records)} runs already done")
else:
    results_records = []
    done_keys = set()

total_experiments = len(EXPERIMENT_MATRIX) * (len(ALPHAS_POS) + len(ALPHAS_NEG))
print(f"\nTotal runs needed: {total_experiments}")
print(f"Already done: {len(done_keys)}")
print(f"Remaining: {total_experiments - len(done_keys)}")
print("=" * 80)

exp_pbar = tqdm(EXPERIMENT_MATRIX.items(), desc="Overall", total=len(EXPERIMENT_MATRIX))

for (vec_name, lang), desc in exp_pbar:
    exp_pbar.set_postfix({'current': desc})

    # ── Literal → Slang (positive alpha) ────────────────────────────────────
    literal_data = data_sample[
        (data_sample['language'] == lang) & (data_sample['label'] == 'Literal')
    ]

    if len(literal_data) > 0:
        for alpha in tqdm(ALPHAS_POS, desc=f"  {desc} Lit→Slang", leave=False):
            if (vec_name, lang, 'Literal→Slang', alpha) in done_keys:
                continue

            preds = classify_batch(
                literal_data['sentence'].tolist(),
                literal_data['term'].tolist(),
                steering_vector=steering_vectors[vec_name],
                alpha=alpha,
                batch_size=BATCH_SIZE_STEERING,
            )

            n_flipped = sum(1 for p in preds if p == 'Slang')
            flip_rate = n_flipped / len(preds)
            n_still_literal = sum(1 for p in preds if p == 'Literal')
            n_unknown = sum(1 for p in preds if p == 'Unknown')

            results_records.append({
                'vector': vec_name,
                'language': lang,
                'direction': 'Literal→Slang',
                'alpha': alpha,
                'n_samples': len(preds),
                'n_flipped': n_flipped,
                'flip_rate': flip_rate,
                'n_unchanged': n_still_literal,
                'n_unknown': n_unknown,
                'description': desc,
            })

            # Print result + examples
            print(f"\n  ✦ {desc} | α={alpha} | Flip rate: {flip_rate:.1%} "
                  f"({n_flipped}/{len(preds)} flipped, {n_unknown} unknown)")
            shown = 0
            for k, p in enumerate(preds):
                if p == 'Slang' and shown < 2:
                    s = literal_data.iloc[k]['sentence'][:80]
                    t = literal_data.iloc[k]['term']
                    print(f"    → [{t}] \"{s}...\" : Literal → Slang ✓")
                    shown += 1

            # Save after every alpha
            pd.DataFrame(results_records).to_csv(STEERING_CHECKPOINT, index=False)

    # ── Slang → Literal (negative alpha) ────────────────────────────────────
    slang_data = data_sample[
        (data_sample['language'] == lang) & (data_sample['label'] == 'Slang')
    ]

    if len(slang_data) > 0:
        for alpha in tqdm(ALPHAS_NEG, desc=f"  {desc} Slang→Lit", leave=False):
            if (vec_name, lang, 'Slang→Literal', alpha) in done_keys:
                continue

            preds = classify_batch(
                slang_data['sentence'].tolist(),
                slang_data['term'].tolist(),
                steering_vector=steering_vectors[vec_name],
                alpha=alpha,
                batch_size=BATCH_SIZE_STEERING,
            )

            n_flipped = sum(1 for p in preds if p == 'Literal')
            flip_rate = n_flipped / len(preds)
            n_still_slang = sum(1 for p in preds if p == 'Slang')
            n_unknown = sum(1 for p in preds if p == 'Unknown')

            results_records.append({
                'vector': vec_name,
                'language': lang,
                'direction': 'Slang→Literal',
                'alpha': alpha,
                'n_samples': len(preds),
                'n_flipped': n_flipped,
                'flip_rate': flip_rate,
                'n_unchanged': n_still_slang,
                'n_unknown': n_unknown,
                'description': desc,
            })

            # Print result + examples
            print(f"\n  ✦ {desc} | α={alpha} | Flip rate: {flip_rate:.1%} "
                  f"({n_flipped}/{len(preds)} flipped, {n_unknown} unknown)")
            shown = 0
            for k, p in enumerate(preds):
                if p == 'Literal' and shown < 2:
                    s = slang_data.iloc[k]['sentence'][:80]
                    t = slang_data.iloc[k]['term']
                    print(f"    → [{t}] \"{s}...\" : Slang → Literal ✓")
                    shown += 1

            # Save after every alpha
            pd.DataFrame(results_records).to_csv(STEERING_CHECKPOINT, index=False)

    # Running summary
    done_so_far = len(results_records)
    print(f"\n  {'─'*60}")
    print(f"  Progress: {done_so_far}/{total_experiments} runs done ({done_so_far/total_experiments:.0%})")

# Final save
results_df = pd.DataFrame(results_records)
results_df.to_csv("/workspace/steering_results.csv", index=False)
print(f"\n{'='*80}")
print(f"✓ ALL DONE — {len(results_df)} steering runs completed")
print(f"Saved to /workspace/steering_results.csv")
print(f"{'='*80}")

CELL STARTED
EXPERIMENT_MATRIX has 12 entries
data_sample has 10653 rows
steering_vectors has 7 keys
✓ Loaded checkpoint: 3 runs already done

Total runs needed: 96
Already done: 3
Remaining: 93


Overall:   0%|          | 0/12 [00:00<?, ?it/s]

  Universal → English Lit→Slang:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 46.49 GiB. GPU 0 has a total capacity of 178.35 GiB of which 21.11 GiB is free. Including non-PyTorch memory, this process has 157.23 GiB memory in use. Of the allocated memory 150.92 GiB is allocated by PyTorch, and 5.55 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ============================================================================
# 9.1 — Summary Table
# ============================================================================

print("\n" + "="*100)
print("STEERING RESULTS SUMMARY")
print("="*100)

for direction in ['Literal→Slang', 'Slang→Literal']:
    print(f"\n{'─'*80}")
    print(f"Direction: {direction}")
    print(f"{'─'*80}")

    subset = results_df[results_df['direction'] == direction]

    pivot = subset.pivot_table(
        values='flip_rate',
        index='description',
        columns='alpha',
        aggfunc='mean'
    )

    # Format as percentages
    display(pivot.style.format('{:.1%}').background_gradient(cmap='RdYlGn', vmin=0, vmax=1))